## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [26]:
from agents import Agent, Runner, trace, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from tavily import TavilyClient
import sendgrid
from sendgrid.helpers.mail import Email, Mail, To, Content
import asyncio
from dotenv import load_dotenv
from IPython.display import display, Markdown
import os

load_dotenv(override=True)

True

In [27]:
# Constants 

ModelName = "gpt-5.4-nano"
NumSearches = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [28]:
TavilyClient = TavilyClient(api_key = os.getenv("TAVILY_API_KEY"))

@function_tool
async def TavilySearch(SearchTerm: str) -> str:
    """ You search the web given a search term and return the results """
    Results = TavilyClient.search(SearchTerm, search_depth="advanced", max_results=NumSearches)
    SearchResults = Results.get("results", [])
    ## Format the search results
    FormattedResults = "\n\n".join([f"Title: {r['title']}\nURL: r['url]\nContent:{r['content']}" for r in SearchResults])
    if FormattedResults is not None:
        return FormattedResults
    else:
        return "No results found"

Instructions = """
               You specialize in web search. You are given a search term, and you must use 
               your TavilySearch tool to search the web for that term and return return the
               summarized result. The TavilySearch tool will give you 5 results for each search
               term. You must summarize these results. Aim for no more than 500-600 words of
               content. Your output will be used by a research assistant to draft a report, so
               it is important to be concise and include important points only. There is no
               need to focus heavily on grammar and producing complete sentence.
               Do not finish till you have produced the summarized result. You must only
               use the results returned by the TavilySearch tool to generate the summary.
               """

SearchAgent = Agent(name="Search Agent", instructions=Instructions, model=ModelName,
                        tools = [TavilySearch], model_settings=ModelSettings(tool_choice="required"))

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [29]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="The reason why this search term is likely to yield results helpful in answering the user query")
    search_term: str = Field(description="The search term to use for web search")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="The list of web searches to be performed to answer the user query")

Instructions = f"""
               Given a query provided by the user, you come up with {NumSearches} search terms, each of which
               will be used to search the web to fetch information pertinent to answering the query. 
               Output {NumSearches} search terms.
               """

PlannerAgent = Agent(name="Planner Agent", instructions=Instructions, model=ModelName,
                          output_type=WebSearchPlan)

## Agent 3: The Writer Agent

In [30]:
Instructions = """
               You are a research assistant who can produce a comprehensive report given 
               a user query. You are provided with the original user query, as well as a
               list of summarized search results generated by another assistant.
               You must only use the summarized search results to generate the report in
               markdown format. 
               First, think of the structure and flow of the report, and the sections that
               should be in it. Revise the flow till the report is very coherent and contains
               all important information.
               The report should be detailed. Aim for at least 1500 words of content.

               Crucial:
               - You must only use the list of summarized search results provided to you to
               synthesize the content. Do not use your own knowledge.
               - Output the report in markdown format            
               """

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the report")
    markdown_report: str = Field(description="The full report in markdown format")
    future_research: str = Field(description="Suggested topics for further research")

ReportAgent = Agent(name="Report Writer", instructions=Instructions, model=ModelName, output_type=ReportData)

## Agent 4: The email agent

In [31]:
@function_tool
def SendEmail(Subject: str, HTMLBody: str):
    """ Send an email to a prospective client """
    SG = sendgrid.SendGridAPIClient(api_key = os.getenv("SENDGRID_API_KEY"))
    FromEmail = Email("spraybubble@gmail.com")
    ToEmail = To("spraybubble@gmail.com")
    content = Content("text/html",HTMLBody)
    mail = Mail(FromEmail, ToEmail, Subject, content).get()
    SG.client.mail.send.post(request_body=mail)
    return {"Mail sent":"OK"}

Instructions = """
               You are en email sender. You are given a research report in markdown format.
               Follow the below steps:
               1. Convert the report into a neatly formatted HTML email
               2. Come up with an appropriate subject line for the email
               3. Use the SendEmail tool to send the email

               Do not finish until you have sent the email using the SendEmail tool
               """

EmailAgent = Agent(name="EmailAgent", instructions=Instructions, model=ModelName,
                        tools = [SendEmail], model_settings=ModelSettings(tool_choice="required"))

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [32]:
async def RunSearches(Query:str) -> list[str]:
    """Plan searches using PlannerAgent and use Search() to search for each search term. Aggregate results into a list."""
    print ("Planning searches....")
    SearchPlan = await Runner.run(PlannerAgent, f"Query: {Query}\n")
    print (f"Number of searches planned: {len(SearchPlan.final_output.searches)}")
    print ("Running searches...\n")
    Tasks = [asyncio.create_task(Search(SearchItem)) for SearchItem in SearchPlan.final_output.searches]
    NumCompleted = 0
    Results = []
    for Task in asyncio.as_completed(Tasks):
        Result = await Task
        if Result:
            NumCompleted +=1
            Results.append(Result)
            print (f"Completed {NumCompleted}/{len(Tasks)} searches")
    print ("\nFinished searching...")
    return Results

async def Search(SearchItem: WebSearchItem) -> str:
    """Run SearchAgent to search for a particular search term and return the summarized results"""
    Input = f"Query: {SearchItem.search_term}\nReason for searching: {SearchItem.reason}"
    Result = await Runner.run(SearchAgent, Input)
    return Result.final_output

The next 2 functions write a report and email it

In [33]:
async def WriteReport(Query:str, SearchResults: list[str]) -> ReportData:
    """Use the ReportAgent to generate the report in markdown format"""
    print ("\nStarting report compilation...")
    Input = f"Original Query: {Query}\n" + "List of summarized results:\n\n"
    Input += "\n\n".join([f"#Search Result {Index}:\n {Result}\n" for Index, Result in enumerate(SearchResults,1)])
    Report = await Runner.run(ReportAgent,Input)
    print ("Finished report compilation...\n")
    return Report.final_output

async def SendReportViaEmail(Report: ReportData) -> str:
    """Use the EmailAgent to send the email"""
    print ("Sending email...")
    await Runner.run(EmailAgent,Report.markdown_report)
    print ("Email sent\n")
    return Report.markdown_report

### Showtime!

In [34]:
Query ="Treatments for severe treatment-refractory OCD as of July, 2026"

with trace("Research trace"):
    print ("Starting research..\n")
    SearchResults = await RunSearches(Query)
    Report = await WriteReport(Query,SearchResults)
    Result = await SendReportViaEmail(Report)
    display(Markdown(Result))
    with open("Report.md","w", encoding = "utf-8") as f:
        f.write(Result)
    print ("Done")

Starting research..

Planning searches....
Number of searches planned: 5
Running searches...

Completed 1/5 searches
Completed 2/5 searches
Completed 3/5 searches
Completed 4/5 searches
Completed 5/5 searches

Finished searching...

Starting report compilation...
Finished report compilation...

Sending email...
Email sent



# Treatments for Severe Treatment-Refractory OCD (as of July 2026) — Evidence Synthesis from Retrieved Sources

## Scope and framing
This report synthesizes the provided search-result summaries on therapies for **severe treatment-refractory obsessive-compulsive disorder (OCD)** up to a **July 2026**-oriented view. The retrieved material covers: 
- **Medication and ERP/CBT optimization** (including augmentation strategies)
- **Neuromodulation** (notably **DBS**, **rTMS/dTMS**, and emerging/adjunct neuromodulation)
- **Ablative lesioning approaches** (e.g., capsulotomy, including **MRgFUS capsulotomy**)
- **Experimental pharmacologic strategies** (notably **ketamine/esketamine**, with psilocybin discussed at a regulatory level but with less OCD-specific clinical detail in the retrieved snippets)

Because the instructions require use of **only the provided search-result summaries**, some topics (e.g., precise July 2026 consensus documents for ketamine+MRgFUS+DBS in one place) are explicitly noted as **not surfaced** in the retrieved results.

---

## 1) Confirming “treatment-refractory” status before escalation
Before declaring OCD “treatment-refractory,” the retrieved evidence emphasizes verifying that the patient has had adequate exposure to core first- and second-line treatments.

### 1.1 Optimize and verify SSRI/SRI trials
A 2023–2025 evidence-focused pharmacotherapy summary stresses that clinicians should confirm:
- **Adherence**
- **Adequate dose**
- **Adequate duration** (with a common timeframe of **~8–12 weeks**)
- **Proper CBT/ERP implementation**

The same source frames escalation logic as: 
1) optimize/ensure an adequate **SSRI first**, potentially using **high-dose** SRI strategies in resistant cases; and if uncertain adherence exists, consider **plasma SSRI levels**.

### 1.2 If still inadequate: switch serotonergic strategies and then augment
If persistent symptoms remain after adequate initial trials:
- **Switch within the SRI class**
- Consider **clomipramine** or **SNRIs**
- Then proceed to **augmentation with a different mechanism**

This “optimize then augment” step is repeatedly implied across the retrieved treatment-refractory framework, and it also connects to why device-based therapies (DBS/capsulotomy) often include patients who have failed both medication and structured CBT/ERP.

---

## 2) Advanced CBT/ERP strategies and medication combination: the strongest “non-device” core
The retrieved evidence shows that the strongest quantitative signal for the psychotherapy+medication combined strategy comes from meta-analytic work.

### 2.1 ERP + pharmacotherapy: key 2024 meta-analytic signal
A **2024 systematic review and meta-analysis** on **exposure and response prevention (ERP) combined with pharmacotherapy** reported:
- **21 RCTs / 1,113 participants**
- **Y-BOCS** as the primary outcome
- Combined treatment superior to control: 
  - **MD = –3.18** (95% CI **–4.97 to –1.40**), **P = 0.0005**
- **High heterogeneity**: **I² = 86%**

**Interpretation within the retrieved material:** the direction favors combined ERP and medication for OCD symptom reduction, but results vary across included RCTs.

### 2.2 D-cycloserine (DCS) augmentation: inconsistent effects
In the same 2024 meta-analytic summary, **DCS augmentation** is described as **inconsistent**. One pediatric RCT (within the review) is described as showing efficacy for **DCS-augmented brief intensive CBT/ERP**, but the review’s overall framing suggests DCS **may not consistently enhance ERP**.

A proposed moderator mentioned in the retrieved summary is whether patients are **taking antidepressants**, which could relate to DCS’s putative fear-extinction mechanism.

### 2.3 “Intensive” ERP: interest but limited directly matching RCT evidence in the retrieved output
The search results describe clinical interest in intensive/advanced CBT/ERP formats and include contextual program-style summaries, plus a broader research direction toward higher intensity exposure schedules. However, within the provided search outputs, a clear 2022–2024 RCT/meta-analysis specifically targeting **treatment-resistant OCD** with an **intensive ERP format** outcome was not identified.

**Practical implication from the retrieved evidence:** while intensive ERP is conceptually appealing and actively discussed, the most concrete pooled evidence returned here is for **ERP + pharmacotherapy** rather than for intensive program format comparisons.

---

## 3) Pharmacotherapy “next steps” for severe treatment-refractory OCD
Once adequate baseline optimization has occurred, the retrieved sources outline augmentation and switching strategies.

### 3.1 High-dose SSRIs and switching to clomipramine
The pharmacotherapy-focused evidence summary highlights:
- SSRIs are first-line, and **higher SRI doses correlate with greater efficacy** (with increased tolerability burden).
- **Clomipramine** is positioned as an effective serotonergic option when SSRI response is inadequate and is used in monotherapy or augmentation.

### 3.2 Antipsychotic augmentation: strongest augmentation class signal in the retrieved material
A 2023–2025 treatment-refractory pharmacotherapy summary identifies atypical antipsychotics as among the best-supported augmentation options.

**Evidence described:**
- A meta-analysis of **14 randomized double-blind placebo-controlled trials** (**n = 491**)
- Conclusion: antipsychotics can be used for augmentation in **SSRI-treated** patients
- Specific agents noted as superior vs placebo: **risperidone, aripiprazole, haloperidol**

**Clinical magnitude described:**
- Response described as **modest** (often around **~one-third**)
- Tolerability limits broader use

**Common dosing ranges noted in the retrieved summary:**
- Risperidone **≤ ~3 mg/day**
- Aripiprazole **≤ ~15 mg/day**

### 3.3 Glutamatergic augmentation: variable signals; NAC has the largest RCT base in the retrieved material
The pharmacotherapy summary reports multiple glutamatergic augmentation options with mixed evidence quality, highlighting that **N-acetylcysteine (NAC)** has the strongest returned RCT support among glutamatergics.

#### NAC (N-acetylcysteine)
- **5 randomized placebo-controlled trials** adding NAC to SSRI monotherapy
- **3** showed greater efficacy vs placebo
- Reported dosing range: **~600 mg to 3000 mg/day** (split dosing in trials)
- Not FDA-regulated (source notes product selection uncertainty)

#### Other glutamatergic agents mentioned with RCT evidence in the retrieved summary
- **Lamotrigine** (2 trials; up to ~100 mg/day)
- **Topiramate** (3 trials; up to ~400 mg/day)
- **Riluzole** (3 trials; up to ~100 mg/day)
- **Memantine** (3 trials; up to ~20 mg/day)
- **Intravenous ketamine** (1 trial; single-dose ~0.5 mg/kg mentioned)

#### Network framing from the retrieved summary
A separate included summary notes that in a larger network-type analysis (as described), **glutamatergic and atypical antipsychotic agents are superior to placebo**, but with **no clear significant differences between them**.

### 3.4 A “next-step” pharmacotherapy algorithm (as synthesized from the retrieved sources)
Based on the retrieved 2023–2025 treatment-refractory summary, a practical escalation path is:
1) **Optimize SSRI** (adequate dose + duration ~8–12 weeks; high-dose strategy if needed; address adherence; consider plasma levels if uncertainty)
2) **If inadequate response:** switch serotonergic strategy (another SSRI → or **clomipramine**; consider SNRI)
3) **If still inadequate:** augment
   - **Antipsychotic augmentation** (risperidone/aripiprazole commonly referenced)
   - or **glutamatergic augmentation**, with **NAC** noted as having the most returned RCT support
4) Continue/ensure **ERP-focused CBT**, and consider combined approaches

The same source also alludes to **somatic interventions** (including rTMS) in the broader algorithm context.

---

## 4) Neuromodulation and advanced procedural options: DBS, capsulotomy/MRgFUS, and TMS
For patients who remain severely incapacitated after maximal guideline-based combined treatment approaches (medication + CBT/ERP), the retrieved material focuses on neuromodulation.

### 4.1 How many remain severe despite best combined efforts?
A real-world/clinical-experience oriented DBS-focused article (post–FDA humanitarian use device approval discussion) reports that:
- Standard pathway typically involves:
  - **CBT with exposure and response prevention (EX/RP)** plus
  - **high-dose serotonergic medication strategies** (SSRIs ± clomipramine and/or antipsychotic augmentation)
- Approximately **~10%** remain severely incapacitated despite maximal recommended combined approaches.
- For these patients, **DBS is discussed as efficacious with acceptable side-effect profile**, though evidence-quality limitations remain (with a need for higher-level studies).
- The article also notes that **programming parameters** are important because real-world consensus may vary in practice.

This provides context for why invasive or semi-invasive neuromodulation appears in treatment-resistant pathways.

---

## 5) Deep Brain Stimulation (DBS) for treatment-refractory OCD
DBS is treated as one of the most concrete procedural options in both consensus/guideline terms (via retrieved guideline update references) and outcome-oriented evidence.

### 5.1 Guideline-style target recommendations (DBS targets)
A key retrieved result anchors DBS target recommendations to formal guideline language:

- A **Congress of Neurological Surgeons (CNS) guideline update** is referenced as an **evidence-based update of the 2014 CNS guidelines**.
- The update includes target-specific recommendations for medically refractory OCD:
  - **Bilateral STN DBS**: “**recommended**” over best medical management for **medically refractory OCD** (described as **Level I evidence basis** language).
  - **Bilateral NAc or BNST DBS**: clinicians **“may use”** (described as **Level II evidence** language).

The retrieved material’s “bottom line” interpretation is that even in the presence of newer research, **formal guideline strength language** can remain anchored to these structured CNS recommendations.

### 5.2 DBS effectiveness and durability signals in the retrieved material
A 2024–2025 neuromodulation synthesis (refractory OCD) reports:
- A **systematic review + meta-analysis of individual participant data** (sham-controlled trials), listed as **Molecular Psychiatry (2024/2025)**, reinforces DBS as effective for treatment-refractory OCD.
- Durability: cohort-style reports of DBS targeting the **ventral ALIC** region show sustained outcomes over years (a “cohort of 50 patients” is mentioned in the retrieved snippet).

Additionally, the CNS-guideline anchored view and the real-world HUDA article emphasize that DBS is a structured next-step but still subject to evidence-quality limits.

### 5.3 Typical DBS candidate selection themes (as summarized)
The 2024–2025 DBS-focused synthesis lists selection criteria themes typically reflected across contemporary literature:
- **Treatment-refractory OCD** defined by failure of adequate trials of guideline-based pharmacotherapy (including high-dose SSRI/clomipramine) and structured CBT/ERP (the refractory paradigm is explicitly mentioned as the logic used in device trials and cohorts).
- Severe symptom burden commonly quantified with **Y-BOCS**
- Chronicity and impairment
- Absence of contraindications to surgery (anesthesia risk, infection risk)
- Ability to participate in **programming and follow-up**

### 5.4 Safety considerations (categories) in the retrieved material
The retrieved DBS synthesis does not provide detailed quantified adverse event rates, but it identifies key risk categories:
- **Surgical hardware-related events** (lead implantation complications, infection, hemorrhage risk)
- **Stimulation-related adverse effects** (mood, dysautonomia, speech/gait changes depending on target and settings)
- **Device maintenance issues over time**

### 5.5 Programming matters
The real-world DBS post–FDA humanitarian use device experience emphasizes that:
- **Programming parameters** are important.
- Because consensus varies, practical outcomes may depend on individualized stimulation settings and programming approach.

---

## 6) Capsulotomy and MR-guided focused ultrasound (MRgFUS) for refractory OCD
In addition to DBS, lesioning strategies appear in the retrieved evidence as comparators or evolving non-invasive options.

### 6.1 Capsulotomy (ablative lesioning) and its conceptual tradeoffs
A 2024–2025 procedural comparison synthesis states:
- Comparative discussion exists historically between **DBS vs anterior capsulotomy**.
- Current debates also include “focused ultrasound capsulotomy” as a non-invasive technique serving as a capsulotomy comparator.

Selection themes mirror DBS logic (refractory OCD with severe Y-BOCS and failed meds/ERP). The retrieved summary highlights practical tradeoffs:
- Lesioning is **irreversible** (“once-and-done”), which can matter for tolerability and psychiatric/cognitive risks.
- Unlike DBS, lesioning lacks the **programmability/reversibility** advantage.

Adverse events categories are described broadly (neurological effects related to targeting and potential psychiatric/cognitive risks such as apathy/hypomania-like effects/emotional blunting), without extracted rates.

### 6.2 MRgFUS capsulotomy: growing evidence signal and durability to ~2 years
Multiple retrieved items highlight MRgFUS capsulotomy:

- A 2024 highlight notes **bilateral MRgFUS capsulotomy** efficacy and safety **up to ~2 years** (described as a South Korea researcher/press-style summary).
- 2024/2025 long-term listings include:
  - *Long-term clinical outcome of a novel bilateral capsulotomy with focused ultrasound in refractory OCD* (Molecular Psychiatry, October 2024 listing), described as providing a durability signal.
  - A 2024 long-term safety/efficacy reporting citation in *Biological Psychiatry* lists OCD and major depressive disorder.
- A 2025 editorial/commentary frames focused ultrasound as a “safe step in the right direction,” while underscoring that broader comparative controlled data are still needed.

Selection constraints implied by MRgFUS procedure nature (from retrieved themes) include suitability for MRI-based targeting and ability to undergo the MR environment for the non-invasive procedure.

### 6.3 Safety categories for MRgFUS capsulotomy (as described)
The retrieved summary mentions typical MRgFUS lesioning adverse event categories at a general level:
- Headache / transducer coupling discomfort
- Transient neurological symptoms
- Risk of edema / lesion-related effects
- Ongoing focus on persistent psychiatric/cognitive effects

As with other procedural summaries, the retrieved snippets do not extract specific incidence rates.

### 6.4 Practical comparative takeaways from the retrieved procedural evidence
The retrieved 2024–2025 procedural comparison synthesis concludes:
- **DBS**: strongest continuity-of-care advantage due to **adjustable programming**, supported by sham-controlled meta-analytic work and long-term cohorts.
- **Capsulotomy**: irreversible; potentially simpler logistics but less flexibility.
- **MRgFUS capsulotomy**: fastest-growing non-invasive “capsulotomy” evidence signal, with reported efficacy/safety to ~2 years and long-term outcome reporting; still needs broader standardized comparative evidence against DBS.

---

## 7) rTMS/dTMS (and other neuromodulation) for treatment-resistant OCD
Among neuromodulation modalities, the retrieved sources identify rTMS/dTMS as having the most actionable evidence specifically for OCD.

### 7.1 rTMS/dTMS: sham-controlled signals and FDA-clearance referenced
The emerging/experimental treatments overview describes:
- rTMS has meaningful, clinically relevant response rates in treatment-resistant OCD with **sham-controlled evidence**.
- Reported effect in the retrieved summary:
  - **~38% response vs ~11% sham**
  - longer-term follow-up “widening the separation” (described qualitatively)

The retrieved material also notes:
- **FDA clearance** for OCD via pivotal trial work in **American Journal of Psychiatry (2019)**.
- Symptom-procedure optimization including **symptom provocation before sessions**.
- A 2025 systematic review is mentioned as suggesting larger effect sizes when **symptom provocation** is used, though the retrieved snippet notes lack of definitive head-to-head confirmation.

### 7.2 tDCS: emerging and less definitive
The same neuromodulation overview notes:
- tDCS is described as part of the evolving toolbox.
- A 2025 review/news release presents a cautious tone: optimistic but not settled.

### 7.3 DBS within neuromodulation: individualized targeting trend
The neuromodulation overview positions DBS as more invasive and complex, while noting movement toward **more individualized targeting** (imaging-based/adaptive approaches), with evidence still developing.

---

## 8) Ketamine and esketamine for refractory OCD: early signals, off-label framing
### 8.1 Mechanistic rationale and “bridging” concept (glutamate/NMDA)
The retrieved evidence describes a mechanistic rationale in a 2025 scoping review:
- OCD is associated with **glutamatergic abnormalities** and **corticostriatal circuit dysfunction**.
- This motivates NMDA-targeting “bridging” effects from **ketamine/esketamine**.

### 8.2 Clinical evidence in the retrieved material: limited, observational/case-level
From the ketamine/esketamine section of the emerging treatments summary:
- Evidence is described as **limited**.
- Adolescent case series/observational reports (2025) are cited with **CY-BOCS improvements**, sometimes only after **repeated infusions**.
- Safety: a case described **psychosis** in the youngest participant, suggesting careful monitoring and possible age/comorbidity considerations.

### 8.3 Regulatory status described in the retrieved material
The retrieved summary states:
- **Esketamine (Spravato)** is FDA-approved for **depression**.
- **Ketamine is not FDA-approved for OCD**.
- Any OCD use is described as **off-label** based on emerging evidence.

### 8.4 Ketamine in the glutamatergic augmentation medication evidence stream
Separately, the pharmacotherapy augmentation summary includes:
- **Intravenous ketamine** (1 trial; single-dose ~0.5 mg/kg mentioned).

Overall, the retrieved materials depict ketamine as biologically plausible but with a far less established OCD evidence base than rTMS/TMS.

---

## 9) Psychedelics (psilocybin) and regulatory motion: OCD-specific evidence unclear in retrieved sources
The emerging treatments overview includes psilocybin primarily through regulatory momentum narratives:
- It references Phase 3 programs (COMPASS programs such as COMP360 / COMP005 & COMP006) and describes 2026 regulatory/accelerated FDA review actions.
- However, it explicitly notes that the retrieved material provides **less concrete OCD-specific Phase 3/approval evidence** and that the detailed trial evidence discussed appears more centered on treatment-resistant depression.

**Conclusion from retrieved material:** psilocybin shows notable regulatory motion by 2026, but OCD-specific efficacy proof is not clearly established in the provided snippets.

---

## 10) Consolidated “what’s most supported” (by evidence maturity within the retrieved sources)
Using only the retrieved summaries, the relative maturity looks like this:

### 10.1 Highest maturity for OCD in retrieved sources
- **CBT/ERP + pharmacotherapy**
  - Strong quantitative support from a 2024 meta-analysis (**ERP + medication superior**, MD –3.18; **I² 86%**).
- **rTMS/dTMS**
  - Sham-controlled response signal (reported **~38% vs 11%** sham) and FDA-cleared OCD via pivotal trial work.
- **DBS**
  - Guideline-anchored target language via CNS guideline update (STN recommended over best medical management; NAc/BNST “may use” based on evidence levels).
  - Sham-controlled evidence synthesis and durability signals.

### 10.2 Promising but earlier-stage signals
- **MRgFUS capsulotomy**
  - Growing efficacy/safety signal with reported durability to ~2 years and long-term outcomes.
  - Comparative head-to-head against DBS still described as needing more controlled data.
- **Ketamine/esketamine**
  - Mechanistic plausibility and early case/observational signals, but limited evidence and off-label framing.

### 10.3 Regulatory momentum but unclear OCD-specific maturity in retrieved sources
- **Psilocybin**
  - Strong regulatory/Phase-3 narratives in general, but less clearly OCD-targeted clinical proof in the retrieved output.

---

## 11) Practical decision framework (synthesized from retrieved evidence themes)
Below is a pragmatic pathway consistent with the retrieved summaries’ logic and escalation thresholds.

### Step 1: Ensure true refractoriness
- Confirm adequate SSRI/SRI dose and duration (**~8–12 weeks**, with possible high-dose)
- Verify adherence; consider plasma SSRI levels if uncertainty
- Confirm adequate CBT/ERP delivery

### Step 2: Combine ERP with optimized medication
- Evidence-supported combined approach (2024 meta-analysis signal)
- Consider medication augmentation (antipsychotics; glutamatergic agents like NAC)

### Step 3: Consider neuromodulation for severe incapacitation
The retrieved material highlights that ~**10%** can remain severely incapacitated despite maximal combined approaches.

- **rTMS/dTMS**: non-invasive neuromodulation with sham-controlled signals and OCD FDA clearance referenced.
- **DBS**: guideline-anchored procedural option for medically refractory OCD, with STN target language “recommended” and NAc/BNST “may use,” plus sham-controlled meta-analytic evidence and durability cohorts.
- **MRgFUS capsulotomy**: increasing evidence, with reported safety/efficacy to ~2 years; irreversible lesioning tradeoffs.
- **Capsulotomy** (non-focused ultrasound variants): exists as an ablative comparator, with irreversible tradeoffs and broad safety categories.

### Step 4: Consider experimental glutamate/NMDA bridging
- **Ketamine/esketamine**: early signals, off-label for OCD, requires careful safety monitoring (including rare serious events noted in the retrieved material).

---

## Conclusion
As of a July 2026-oriented synthesis based on the provided results, severe treatment-refractory OCD management is anchored by:
- **optimized ERP/CBT with medication**, where a 2024 meta-analysis reports superiority for ERP + pharmacotherapy;
- escalation to **neuromodulation** when severe impairment persists (noted to occur in ~10% despite maximal combined standard care), with **rTMS/dTMS** offering the strongest non-invasive evidence signal (FDA-cleared, sham-controlled response separation) and **DBS** offering the most structured guideline-style target recommendations (CNS guideline update language) supported by sham-controlled synthesis and durability reports.

Procedural lesioning with **MRgFUS capsulotomy** shows an emerging and growing evidence base (efficacy/safety up to ~2 years in the retrieved material), while **ketamine/esketamine** appears promising but remains early and off-label for OCD in the retrieved summaries.

---

## Future Research (suggested based on gaps explicitly or implicitly reflected in the retrieved sources)
1. **Head-to-head comparative studies**: standardized comparisons of **DBS vs MRgFUS capsulotomy** in refractory OCD populations (the retrieved material emphasizes need for broader controlled comparative evidence).
2. **Mechanism-informed prediction**: further work on **predictors/biomarkers** for DBS response (the retrieved sources mention ongoing refinement including predictors and biomarkers).
3. **Optimization of neuromodulation parameters**: comparative evaluation of **DBS programming strategies** and **TMS symptom-provocation protocols** (the retrieved material highlights that procedural optimization matters, but definitive head-to-head evidence is limited).
4. **Ketamine/esketamine safety and efficacy trials in OCD**: the retrieved evidence is limited and includes a severe safety signal (psychosis in a young participant). Future RCTs should clarify dosing schedules, monitoring protocols, and patient selection.
5. **Intensive ERP format RCTs in refractory OCD**: the retrieved results acknowledge clinical interest but do not surface a dedicated 2022–2024 intensive ERP RCT/meta-analysis for refractory OCD.
6. **Consolidated July 2026 consensus**: efforts to update integrated guidance across modalities (ERP/meds/neuromodulation/ketamine) are not clearly present in the retrieved outputs; research should support clearer consensus frameworks.


Done


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>